<div style="text-align: center;">

# Hazard Data API Basics: Request, Transform, and Visualize Point-Level Climate Data

</div>

This notebook is a practical introduction to the Alpha-Klima hazard-data API. It shows how to request climate hazard intensities for a set of latitude/longitude points, reshape the nested API response into a table, and visualize one result slice on a map.

You can use this notebook as a template when you want to integrate hazard data directly into a Python workflow before moving on to asset-impact or portfolio-level notebooks.

The workflow is:

1. Load API credentials from the local environment.
2. Ask the API which hazard sources, indicators, scenarios, and years are available.
3. Create a small example set of point locations around London.
4. Build a hazard-data request for coastal flood depth across scenarios and years.
5. Submit the request to `/api/get_hazard_data`.
6. Inspect the response shape and basic intensity ranges.
7. Convert nested intensity curves into a return-period table.
8. Render an interactive map for one scenario and year.

The example requests `CoastalInundation` with the `flood_depth` indicator. The same structure applies to other hazards and indicators returned by the availability endpoint.

## 1. Imports

When you run the notebook for the first time, select `.venv` as the interpreter. If you are unable to find the `.venv` interpreter, please follow the *Setup* instructions in the `README.md` to set it up.

The notebook uses a small set of common Python packages:

- `requests` sends HTTPS requests to the Alpha-Klima API.
- `numpy` creates a reproducible set of example locations.
- `pandas` turns nested JSON into tables that are easier to inspect and export.
- `plotly` creates an interactive point map.
- `python-dotenv` loads credentials from `.env` so secrets are not hard-coded in the notebook.

In [1]:
import os

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import requests
from dotenv import load_dotenv
from IPython.display import HTML, display

In [2]:
# Render Plotly figures in both live notebooks and static notebook previews.
import plotly.io as pio

pio.renderers.default = "notebook+png"

## 2. Load API Configuration

The API base URL and API key are read from `../../.env`, which is two levels above this notebook. If you have not configured the `.env` file, please follow the *API credentials* instructions in the `README.md` to set it up.

Expected environment variables:

- `ALPHA_KLIMA_API_KEY`: API key used to authenticate requests.
- `ALPHA_KLIMA_API_BASE_URL`: base URL of the Alpha-Klima API, without a trailing slash requirement.

The API key is sent in the `X-API-Key` header. Keeping this setup in one cell makes it easier to reuse the same credentials for several endpoint calls.

In [3]:
_ = load_dotenv("../../.env")
API_KEY = os.environ.get("ALPHA_KLIMA_API_KEY")
API_BASE = os.environ.get("ALPHA_KLIMA_API_BASE_URL", "").rstrip("/")

if not API_KEY or not API_BASE:
    raise RuntimeError(
        "Set ALPHA_KLIMA_API_KEY and ALPHA_KLIMA_API_BASE_URL in ../../.env before running this notebook."
    )

HEADERS = {"X-API-Key": API_KEY, "Accept": "application/json"}
print("API base:", API_BASE)

API base: https://platform.alpha-klima.com/prapi


## 3. Define a Small API Helper

The API returns JSON for successful calls and an HTTP error for failed calls. This helper keeps request handling consistent across the notebook:

- it joins the base URL and endpoint path,
- sends the same authentication headers every time,
- raises an exception when the API returns an error status,
- and returns parsed JSON for successful responses.

In [4]:
def api_post(endpoint: str, payload: dict, timeout: int = 240) -> dict:
    """POST a JSON payload to the Alpha-Klima API and return the JSON response."""
    response = requests.post(
        f"{API_BASE}{endpoint}",
        json=payload,
        headers=HEADERS,
        timeout=timeout,
    )
    response.raise_for_status()
    return response.json()

## 4. Discover Available Hazard Sources

Before requesting hazard values, first ask `/api/get_available_sources` what is available to your API key and use case.

The response is grouped by hazard family and indicator. For each indicator, it lists available scenarios and years. This matters because a request must use combinations that exist in the source inventory. For example, a flood dataset may support `rcp4p5` in 2035 and 2085, while a heat dataset may use SSP scenarios such as `ssp585`.

This notebook uses `use_case_id="ECB_SCORES"`, matching the current examples in this repository. If your API access is configured for a different use case, update the request accordingly.

In [5]:
availability_request = {
    "include_all": False,
    "use_case_id": "API_EXAMPLE",
    "selected_hazards_list": [],
}

dict_available_sources = api_post("/api/get_available_sources", availability_request)

In [6]:
def available_sources_to_table(available_sources: dict) -> pd.DataFrame:
    """Flatten the availability response into one row per hazard indicator."""
    rows = []

    for hazard_type, indicators in available_sources.get("hazards", {}).items():
        for indicator_id, metadata in indicators.items():
            scenarios = [
                {"scenario": scenario["id"], "years": scenario.get("years", [])}
                for scenario in metadata.get("scenarios", [])
            ]
            rows.append(
                {
                    "hazard_type": hazard_type,
                    "indicator_id": indicator_id,
                    "indicator_display_name": metadata.get("indicator_display_name", ""),
                    "scenario_years": scenarios,
                    "available_for_asset_types": metadata.get("available_for_assets_type", []),
                }
            )

    return pd.DataFrame(rows).sort_values(["hazard_type", "indicator_id"]).reset_index(drop=True)


available_sources_df = available_sources_to_table(dict_available_sources)
available_sources_df.head(20)

,hazard_type,indicator_id,indicator_display_name,scenario_years,available_for_asset_types
0,ChronicHeat,cooling_degree_days/index,Cooling Degree Days (Alpha-Klima),"[{'scenario': 'historical', 'years': [2005]}, ...","[TelecommunicationAsset, RealEstateAsset]"
1,ChronicHeat,days_wbgt_above,Days With Wet-Bulb Globe Temperature Above Thr...,"[{'scenario': 'historical', 'years': [2005]}, ...",[ManufacturingAsset]
2,ChronicHeat,mean_degree_days/above/32c,Mean Degree Days Above 32°C (ACCESS-CM2),"[{'scenario': 'historical', 'years': [2005]}, ...",[IndustrialActivity]
3,ChronicHeat,mean_work_loss/high,"Mean Work Loss, high Intensity (ACCESS-CM2)","[{'scenario': 'historical', 'years': [2005]}, ...",[IndustrialActivity]
4,CoastalInundation,flood_depth,Coastal Flood Depth (RAIN Project),"[{'scenario': 'historical', 'years': [1971]}, ...","[InfrastructureAsset, IndustrialActivity, Util..."
5,Drought,cdd,Consecutive Dry Days (Copernicus),"[{'scenario': 'historical', 'years': [1995]}, ...","[InfrastructureAsset, IndustrialActivity, Tele..."
6,Drought,spi6,"Standardized Precipitation Index, SPI-6 (Coper...","[{'scenario': 'historical', 'years': [1995]}, ...","[InfrastructureAsset, IndustrialActivity, Tele..."
7,Earthquake,pga_10pc50yr,"Peak Ground Acceleration, 10% Exceedance in 50...","[{'scenario': 'historical', 'years': [2023]}]","[InfrastructureAsset, TelecommunicationAsset, ..."
8,Fire,fire_probability,Annual Wildfire Probability (Alpha-Klima),"[{'scenario': 'historical', 'years': [2010]}, ...","[InfrastructureAsset, IndustrialActivity, Util..."
9,FreezingRain,freezing_rain_probability,Annual Freezing Rain Probability (RAIN Project),"[{'scenario': 'historical', 'years': [1985]}, ...","[InfrastructureAsset, IndustrialActivity, Tele..."


## 5. Define Locations of Interest

The hazard-data endpoint works with point locations. Each request item contains a list of latitudes and a list of longitudes. The two lists must have the same length and matching order: the first latitude is paired with the first longitude, the second latitude with the second longitude, and so on.

For demonstration, we create 50 reproducible random points around London. In a client workflow, replace this synthetic sample with coordinates from an asset register, exposure dataset, CSV file, database query, or GIS pipeline.

Coordinates are expected in EPSG:4326, which is the standard latitude/longitude coordinate reference system used by GPS and most web maps.

In [7]:
# Center point near London. Longitude is positive here because the sample area is east of Greenwich.
center_lat = 51.5155
center_lon = 0.0495

# Approximate 5-6 km north/south and east/west bounding box around the center point.
dlat = 0.05
dlon = 0.08

rng = np.random.default_rng(7)
n_points = 50

points = pd.DataFrame(
    {
        "lat": rng.uniform(center_lat - dlat, center_lat + dlat, n_points),
        "lon": rng.uniform(center_lon - dlon, center_lon + dlon, n_points),
    }
)

points.head()

,lat,lon
0,51.528010,0.027302
1,51.555221,0.065209
2,51.543069,-0.021020
3,51.488021,0.031521
4,51.495517,0.021186


## 6. Build the Hazard Data Request

The `/api/get_hazard_data` endpoint expects an `items` list. Each item asks for one hazard, indicator, scenario, and year combination evaluated at the same set of points.

Important fields:

- `request_item_id`: your own label for matching a response item back to the request item.
- `hazard_type`: hazard family, such as `CoastalInundation`, `RiverineInundation`, or `ChronicHeat`.
- `indicator_id`: measure requested for that hazard, such as `flood_depth`.
- `scenario`: climate scenario, such as `historical`, `rcp4p5`, or `rcp8p5`.
- `year`: historical baseline or future horizon. In the underlying API schema, the exact historical year is not material for the `historical` scenario, but keeping a baseline year in the request makes the example easier to read.
- `latitudes` and `longitudes`: point coordinates to evaluate.

In [8]:
request_specs = [
    ("coastal_rcp4p5_2035", "rcp4p5", 2035),
    ("coastal_rcp4p5_2085", "rcp4p5", 2085),
    ("coastal_rcp8p5_2035", "rcp8p5", 2035),
    ("coastal_rcp8p5_2085", "rcp8p5", 2085),
    ("coastal_historical", "historical", 1971),
]

request_dict = {
    "items": [
        {
            "request_item_id": request_item_id,
            "hazard_type": "CoastalInundation",
            "indicator_id": "flood_depth",
            "scenario": scenario,
            "year": year,
            "latitudes": points["lat"].to_list(),
            "longitudes": points["lon"].to_list(),
        }
        for request_item_id, scenario, year in request_specs
    ]
}

request_dict["items"][0]

{'request_item_id': 'coastal_rcp4p5_2035',
 'hazard_type': 'CoastalInundation',
 'indicator_id': 'flood_depth',
 'scenario': 'rcp4p5',
 'year': 2035,
 'latitudes': [51.52800954666047,
  51.55522138009696,
  51.54306856902452,
  51.48802071899907,
  51.495516628491124,
  51.55285534453963,
  51.46602653045656,
  51.547622841838276,
  51.545206942875204,
  51.512293495284375,
  51.49580324268194,
  51.493342561210085,
  51.49098695876542,
  51.51000763058827,
  51.5159548258958,
  51.52084973520745,
  51.56505002834344,
  51.544766191921376,
  51.52771792294412,
  51.564396014768185,
  51.48703086982356,
  51.48152120338579,
  51.52675396042731,
  51.46989420079614,
  51.46906802787736,
  51.51698888202714,
  51.512120602532534,
  51.55721677731928,
  51.5284226254491,
  51.51691176465995,
  51.51518734353935,
  51.49025149220274,
  51.466679402554256,
  51.484740214398535,
  51.53470321208819,
  51.485560672398705,
  51.50245363106023,
  51.46587342420521,
  51.54850477298017,
  51.4809

## 7. Execute the Hazard Data Request

The response contains an `items` list that mirrors the request. For each response item, the main payload is `intensity_curve_set`:

- There is one curve per requested point.
- `index_values` contains the curve index. For acute hazards such as flood and wind, these are return periods in years.
- `intensities` contains hazard intensity values aligned with those index values. In this example, intensities are coastal flood depths in meters.
- `index_name` describes the meaning of `index_values`. For acute hazards it is usually `return period`; for some chronic indicators it may be `threshold`.

A return period is a frequency concept. A 100-year return-period flood depth corresponds to an annual exceedance probability of roughly `1 / 100 = 1%` under the modeled conditions. It does not mean the event can happen only once every 100 years.

In [9]:
dict_hazard_data = api_post("/api/get_hazard_data", request_dict)

In [10]:
def summarize_hazard_response(data: dict) -> pd.DataFrame:
    """Create a compact sanity-check table for the nested hazard response."""
    rows = []

    for item in data.get("items", []):
        curves = item.get("intensity_curve_set", [])
        values = [
            value
            for curve in curves
            for value in curve.get("intensities", [])
            if value is not None
        ]
        rows.append(
            {
                "request_item_id": item.get("request_item_id"),
                "hazard_type": item.get("hazard_type") or item.get("event_type"),
                "indicator_id": item.get("indicator_id"),
                "scenario": item.get("scenario"),
                "year": item.get("year"),
                "locations_returned": len(curves),
                "min_intensity": min(values) if values else np.nan,
                "max_intensity": max(values) if values else np.nan,
            }
        )

    return pd.DataFrame(rows)


response_summary = summarize_hazard_response(dict_hazard_data)
response_summary

,request_item_id,hazard_type,indicator_id,scenario,year,locations_returned,min_intensity,max_intensity
0,coastal_rcp4p5_2035,CoastalInundation,flood_depth,rcp4p5,2035,50,0.0,5.194
1,coastal_rcp4p5_2085,CoastalInundation,flood_depth,rcp4p5,2085,50,0.0,5.149
2,coastal_rcp8p5_2035,CoastalInundation,flood_depth,rcp8p5,2035,50,0.0,4.793
3,coastal_rcp8p5_2085,CoastalInundation,flood_depth,rcp8p5,2085,50,0.0,5.193
4,coastal_historical,CoastalInundation,flood_depth,historical,1971,50,0.0,5.034


## 8. Convert Intensity Curves to a Return-Period Table

The raw API response is nested because each request item contains many point-level curves. For analysis, export, or visualization, it is often easier to work with a flat table.

The helper below creates one row per point, scenario, and year. It selects the return periods used most often in this example: 10, 30, 100, 300, and 1000 years. If a requested return period is not present in a returned curve, the corresponding table value is set to `NaN`.

This transformation does not change the hazard values. It only reshapes the API response into a table.

In [11]:
def build_return_period_table(
    data: dict,
    points: pd.DataFrame,
    return_periods: tuple[float, ...] = (10.0, 30.0, 100.0, 300.0, 1000.0),
) -> pd.DataFrame:
    """Flatten API response curves into one row per point, scenario, and year."""
    rows = []

    for item in data.get("items", []):
        curves = item.get("intensity_curve_set", [])
        if len(curves) != len(points):
            print(
                f"Warning: {item.get('request_item_id')} returned {len(curves)} curves "
                f"for {len(points)} input points. Rows will be built for matched pairs only."
            )

        for point, curve in zip(points.to_dict("records"), curves):
            intensity_by_return_period = dict(
                zip(curve.get("index_values", []), curve.get("intensities", []))
            )
            row = {
                "lat": point["lat"],
                "lon": point["lon"],
                "request_item_id": item.get("request_item_id"),
                "hazard_type": item.get("hazard_type") or item.get("event_type"),
                "indicator_id": item.get("indicator_id"),
                "scenario": item.get("scenario"),
                "year": item.get("year"),
            }
            for return_period in return_periods:
                row[f"rp{int(return_period)}"] = intensity_by_return_period.get(
                    return_period,
                    np.nan,
                )
            rows.append(row)

    return pd.DataFrame(rows)


df_rp = build_return_period_table(dict_hazard_data, points)
df_rp.head()

,lat,lon,request_item_id,hazard_type,indicator_id,scenario,year,rp10,rp30,rp100,rp300,rp1000
0,51.528010,0.027302,coastal_rcp4p5_2035,CoastalInundation,flood_depth,rcp4p5,2035,NaN,NaN,0.000,NaN,NaN
1,51.555221,0.065209,coastal_rcp4p5_2035,CoastalInundation,flood_depth,rcp4p5,2035,NaN,NaN,0.000,NaN,NaN
2,51.543069,-0.021020,coastal_rcp4p5_2035,CoastalInundation,flood_depth,rcp4p5,2035,NaN,NaN,0.000,NaN,NaN
3,51.488021,0.031521,coastal_rcp4p5_2035,CoastalInundation,flood_depth,rcp4p5,2035,NaN,NaN,NaN,NaN,0.5306
4,51.495517,0.021186,coastal_rcp4p5_2035,CoastalInundation,flood_depth,rcp4p5,2035,3.666,3.916,4.185,4.428,4.6940


In [12]:
df_rp.groupby(["scenario", "year"])[["rp10", "rp30", "rp100", "rp300", "rp1000"]].describe()

rp10                                                     \
                count     mean       std     min      25%     50%    75%   
scenario   year                                                            
historical 1971  10.0  2.49911  1.154565  0.4521  2.00525  2.2065  3.386   
rcp4p5     2035  10.0  2.60411  1.154565  0.5571  2.11025  2.3115  3.491   
           2085  10.0  2.78011  1.154565  0.7331  2.28625  2.4875  3.667   
rcp8p5     2035  10.0  2.47335  1.154408  0.4265  1.98000  2.1810  3.360   
           2085  10.0  2.82311  1.154565  0.7761  2.32925  2.5305  3.710   

                        rp30           ...    rp300        rp1000            \
                   max count     mean  ...      75%    max  count      mean   
scenario   year                        ...                                    
historical 1971  4.061  10.0  2.73611  ...  3.58200  4.782   14.0  2.729570   
rcp4p5     2035  4.166  10.0  2.85411  ...  3.72800  4.928   15.0  2.732307   
           2085  4.342  10.0  2.97711  ...  3.74000  4.940   15.0  2.687307   
rcp8p5     2035  4.035  10.0  2.65811  ...  3.92200  4.597   13.0  2.694623   
           2085  4.385  10.0  3.02011  ...  3.62675  4.984   15.0  2.731307   

                                                                     
                      std      min      25%     50%      75%    max  
scenario   year                                                      
historical 1971  1.661038  0.05358  1.67575  3.0365  3.67675  5.034  
rcp4p5     2035  1.712585  0.21360  1.06285  3.0800  3.67950  5.194  
           2085  1.712585  0.16860  1.01785  3.0350  3.63450  5.149  
rcp8p5     2035  1.531589  0.10600  2.18700  2.9120  3.59300  4.793  
           2085  1.712585  0.21260  1.06185  3.0790  3.67850  5.193  

[5 rows x 40 columns]

## 9. Visualize One Scenario-Year Slice

The map helper below filters the return-period table to one `(scenario, year)` slice and maps the selected point locations. It uses marker size and color to show hazard intensity at different return periods.

Encoding choices:

- Marker size uses `rp10`, with fallbacks to `rp30` and then `rp100`. This emphasizes more frequent flooding.
- Marker color uses `rp100`, with fallbacks to `rp300` and then `rp30`. This emphasizes a more severe but still commonly reported return period.

These choices are illustrative. In a real analysis, choose the return period that best matches the decision question, risk appetite, regulation, or engineering design standard.

In [13]:
def coastal_flood_depth_map(
    df_rp: pd.DataFrame,
    scenario: str,
    year: int,
    map_style: str = "open-street-map",
    zoom: float = 11.5,
    height: int = 520,
) -> go.Figure:
    """Map point-level coastal flood depths for one scenario and year."""
    color_scale = [[0, "#3A84A0"], [0.5, "#0092A0"], [1, "#E15E0B"]]
    subset = df_rp[(df_rp["scenario"] == scenario) & (df_rp["year"] == year)].copy()

    if subset.empty:
        raise ValueError(f"No rows found for scenario={scenario!r}, year={year!r}.")

    size_value = subset["rp10"].where(
        subset["rp10"].notna(),
        subset["rp30"].where(subset["rp30"].notna(), subset["rp100"]),
    )
    color_value = subset["rp100"].where(
        subset["rp100"].notna(),
        subset["rp300"].where(subset["rp300"].notna(), subset["rp30"]),
    )

    subset["_marker_size"] = size_value.fillna(0).clip(lower=0)
    subset["_marker_color"] = color_value.fillna(0).clip(lower=0)

    fig = px.scatter_map(
        subset,
        lat="lat",
        lon="lon",
        color="_marker_color",
        size="_marker_size",
        color_continuous_scale=color_scale,
        size_max=18,
        zoom=zoom,
        height=height,
        hover_data={
            "request_item_id": True,
            "rp10": ":.2f",
            "rp30": ":.2f",
            "rp100": ":.2f",
            "rp300": ":.2f",
            "rp1000": ":.2f",
            "_marker_size": False,
            "_marker_color": False,
        },
    )
    fig.update_layout(
        map_style=map_style,
        map_center={"lat": float(subset["lat"].mean()), "lon": float(subset["lon"].mean())},
        title=(
            "Coastal inundation flood depth"
            f"<br><span style='font-size:12px'>{scenario}, {year}. "
            "Size: RP10 depth | Color: RP100 depth</span>"
        ),
        coloraxis_colorbar={"title": "RP100 depth"},
        font={"family": "Source Sans Pro, Open Sans, Arial", "size": 12, "color": "#004D73"},
        title_font={"size": 14, "family": "Source Sans Pro, Open Sans, Arial", "color": "#0092A0"},
        margin={"l": 10, "r": 10, "t": 70, "b": 10},
        paper_bgcolor="white",
    )
    return fig

In [14]:
def show_plotly(fig: go.Figure) -> None:
    """Display a Plotly figure in notebooks that may not load the default renderer."""
    html = pio.to_html(fig, include_plotlyjs="cdn", full_html=False)
    display(HTML(html))

### Render the Example Map

The final cell renders the map for `rcp4p5` in 2035. Change `scenario`, `year`, or `map_style` to compare other request items.

For client notebooks, `open-street-map` is a reliable default basemap because it does not require an additional map token.

In [15]:
fig = coastal_flood_depth_map(
    df_rp,
    scenario="rcp4p5",
    year=2035,
    map_style="open-street-map",
)

show_plotly(fig)

![Figure](demo_figures/figure1.png)

## 10. Where This Fits in the Broader Workflow

This notebook stops at hazard intensity: it answers questions such as "what modeled flood depth is associated with this point under this scenario and return period?"

Portfolio risk workflows usually add two more layers:

1. Asset context, such as asset class, location, value, occupancy, floor height, or protection assumptions.
2. Vulnerability or impact modeling, which translates hazard intensity into expected damage, loss, disruption, or financial metrics.

Use this notebook when you need direct access to hazard indicators. Use the asset-impact notebooks when you want Alpha-Klima to combine hazard data with asset characteristics and vulnerability models.